# 02 · 2AFC (siyam) eğitimi ve iki sıra testi

Kurulum → eğitim (rejim × seed listesi) → kayıp ve doğruluk eğrileri → iki sıra testi → özet.
**Önce:** Runtime → Change runtime type → T4 GPU. Her eğitim yaklaşık 12 dakika.
Bağlantı koparsa not defterini baştan çalıştırın: biten eğitimler atlanır, yarım kalanlar yeniden başlar.

In [ ]:
import torch, platform
print("PyTorch", torch.__version__, "| Python", platform.python_version())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK → Runtime > Change runtime type > T4 GPU")

In [ ]:
GITHUB_URL = "https://github.com/zserraoktutan/ebbinghaus-human-ai.git"
DAL = "main"

import os, sys
from google.colab import drive
drive.mount("/content/drive")

PROJE = "/content/ebbinghaus-human-ai"
if not os.path.exists(PROJE):
    !git clone -q -b {DAL} {GITHUB_URL} {PROJE}
else:
    !git -C {PROJE} pull -q origin {DAL}

SONUC = "/content/drive/MyDrive/ebbinghaus-sonuclar/results"   # ortak klasör
os.makedirs(SONUC, exist_ok=True)
os.chdir(PROJE)
if PROJE not in sys.path:
    sys.path.insert(0, PROJE)
print("Proje :", PROJE)
print("Sonuç :", SONUC)

In [ ]:
# Çalıştırılacak eğitimler: (rejim, seed). P3 ana matrisi: A ve B-fiz × seed 0, 1, 2
CALISTIR = [("A", 0), ("B-fiz", 0), ("A", 1), ("B-fiz", 1), ("A", 2), ("B-fiz", 2)]
EPOCHS = 10        # plan: 10
N_PER_CELL = 50    # plan: her (koşul, boyut) hücresinde 50 görsel

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from model.train import run_name
from model.train_2afc import train_2afc, load_2afc
from model.test_2afc import order_test


def egitim_tamam_mi(ad):
    """Log dosyasında EPOCHS kadar satır varsa eğitim bitmiştir."""
    log = f"{SONUC}/{ad}_log.csv"
    return os.path.exists(log) and len(pd.read_csv(log)) >= EPOCHS


def egri_ciz(hist, ad):
    ep = [h["epoch"] for h in hist]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    ax[0].plot(ep, [h["train_loss"] for h in hist], "o-", label="eğitim")
    ax[0].plot(ep, [h["val_loss"] for h in hist], "s-", label="doğrulama")
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("BCE kaybı"); ax[0].legend(); ax[0].set_title(f"{ad}: kayıp")
    ax[1].plot(ep, [h["train_acc"] for h in hist], "o-", label="eğitim")
    ax[1].plot(ep, [h["val_acc"] for h in hist], "s-", label="doğrulama")
    ax[1].axhline(50, ls="--", color="gray")
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("doğruluk (%)"); ax[1].legend(); ax[1].set_title(f"{ad}: doğruluk")
    plt.tight_layout()
    plt.savefig(f"{SONUC}/{ad}_egriler.png", dpi=150)
    plt.show()


for rejim, seed in CALISTIR:
    ad = run_name("2afc", rejim, seed)
    tablo = f"{SONUC}/{ad}_siratesti.csv"

    if os.path.exists(tablo):
        print(f"{ad}: test tablosu zaten var, atlanıyor")
        continue

    if egitim_tamam_mi(ad):
        net, _ = load_2afc(f"{SONUC}/{ad}.pt")
        print(f"{ad}: eğitilmiş model yüklendi")
    else:
        net, hist = train_2afc(rejim, seed=seed, out_dir=SONUC, epochs=EPOCHS)
        egri_ciz(hist, ad)

    df = pd.DataFrame(order_test(net, n_per_cell=N_PER_CELL))
    df.to_csv(tablo, index=False)
    print(f"{ad}: iki sıra testi kaydedildi, ortalama taraf yanlılığı {df['bias'].mean():+.3f}\n")

In [ ]:
import glob

parcalar = []
for f in sorted(glob.glob(f"{SONUC}/model_2afc_*_siratesti.csv")):
    ad = os.path.basename(f).replace("_siratesti.csv", "")
    parcalar.append(pd.read_csv(f).assign(model=ad))
tum = pd.concat(parcalar)

print("p (karşılaştırma = 1.00), sıra yanlılığından arındırılmış:")
print(tum[tum["comparison"] == 1.0].pivot(index="condition", columns="model", values="p").round(3))
print("\nOrtalama taraf yanlılığı:")
print(tum.groupby("model")["bias"].mean().round(3))